# MLflow Session-Level Evaluation Demo: Casper's Kitchen Order Support

![LLM as Judge](https://raw.githubusercontent.com/dmatrix/mlflow-misc/main/genai/agents/multi_turn/images/llm_as_judge.png)

This notebook demonstrates MLflow session-level evaluation capabilities using a multi-turn
customer support agent for **Casper's Kitchen**, a catering company.

The agent handles order issues (late deliveries, wrong items, allergy concerns) across
multiple conversation turns, and we evaluate the full conversation using MLflow judges.

## What You'll Learn

- How to register prompts in the **MLflow Prompt Registry** (Unity Catalog-backed)
- How to build a multi-turn agent with MLflow session tracking
- How to create **session-level judges** using the `{{ conversation }}` template
- How to run `mlflow.genai.evaluate()` on full conversations
- How to evaluate **coherence** and **context retention** across turns

## Requirements

- Databricks workspace with Foundation Model API access
- A Unity Catalog `catalog.schema` with write access (for Prompt Registry)
- No external API keys needed

---

In [0]:
%pip install mlflow==3.10.1 openai litellm==1.82.6 databricks-agents databricks-sdk

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
print(f"MLflow version: {mlflow.__version__}")

## Configuration

Credentials are auto-detected from the Databricks notebook context.

**One thing to set:** the `PROMPT_CATALOG` and `PROMPT_SCHEMA` below must point to a
Unity Catalog catalog and schema where you have write access. This is where the
MLflow Prompt Registry will store versioned prompts.

In [0]:
import os

# Auto-detect current user
USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)

# ============================================================
# >>> SET THESE to a catalog.schema you have write access to <<<
# ============================================================
PROMPT_CATALOG = "shm"
PROMPT_SCHEMA = "ml"

# Model configuration
AGENT_MODEL = "databricks-gpt-5"
JUDGE_MODEL = "databricks-gemini-2-5-flash"
EXPERIMENT_NAME = "caspers-kitchen-order-support"

# Databricks credentials (auto-populated from notebook context)
os.environ["DATABRICKS_HOST"] = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiUrl().get()
)
os.environ["DATABRICKS_TOKEN"] = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)

# MLflow judges use litellm internally, which reads these env vars
# to reach Databricks-served models via the OpenAI-compatible API
os.environ["OPENAI_API_KEY"] = os.environ["DATABRICKS_TOKEN"]
os.environ["OPENAI_API_BASE"] = f"{os.environ['DATABRICKS_HOST']}/serving-endpoints"

# Fully qualified prompt name prefix
PROMPT_PREFIX = f"{PROMPT_CATALOG}.{PROMPT_SCHEMA}"

print(f"User:           {USER_EMAIL}")
print(f"Agent model:    {AGENT_MODEL}")
print(f"Judge model:    {JUDGE_MODEL}")
print(f"Prompt prefix:  {PROMPT_PREFIX}")
print(f"Experiment:     /Users/{USER_EMAIL}/{EXPERIMENT_NAME}")
print(f"DB Host:        {os.environ['DATABRICKS_HOST']}")

In [0]:
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(f"/Users/{USER_EMAIL}/{EXPERIMENT_NAME}")
mlflow.openai.autolog()

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment:   /Users/{USER_EMAIL}/{EXPERIMENT_NAME}")

## Step 1: Register Prompts in the MLflow Prompt Registry

Prompts are defined as templates, then registered in the Unity Catalog-backed
MLflow Prompt Registry. Once registered, they are **versioned** and can be edited
via the MLflow UI without changing notebook code.

The agent and judges will load prompts from the registry at runtime using
`mlflow.genai.load_prompt()`.

In [0]:
import mlflow

# --- Prompt Templates ---

SYSTEM_PROMPT_TEMPLATE = """You are a helpful customer support agent for Casper's Kitchen,
a catering company that delivers meals for office events, meetings, and parties.

Your responsibilities:
- Help customers with order issues: late deliveries, wrong items, missing items
- Handle allergy and dietary concerns with urgency and care
- Process refund and credit requests
- Answer menu and event catering questions
- Escalate when appropriate

Guidelines:
- Keep responses under 100 words
- Be empathetic, professional, and solution-oriented
- Remember ALL details the customer has told you across turns:
  order numbers, addresses, allergy info, timing, prior complaints
- Never re-ask for information already provided
- Reference previous messages when relevant
- Treat allergy concerns as urgent safety issues
"""

COHERENCE_JUDGE_TEMPLATE = """Evaluate the coherence of this multi-turn customer support conversation.

{{ conversation }}

Does the conversation flow logically across turns?
- Are responses relevant to what was asked?
- Does the agent avoid contradicting itself?
- Does the agent maintain a professional, helpful tone throughout?

Value: True if coherent, False if there are significant coherence issues.
Rationale: 2-3 sentences on flow, consistency, and helpfulness.
"""

CONTEXT_RETENTION_JUDGE_TEMPLATE = """Evaluate context retention in this multi-turn customer support conversation.

{{ conversation }}

The agent should remember key details from earlier turns:
order numbers, delivery addresses, allergy information, timing details,
prior complaints, and any other customer-provided context.

EXCELLENT: All prior details applied automatically in every relevant turn.
GOOD: Most context retained; minor lapses that don't derail the conversation.
FAIR: Occasionally re-asks for info already given; forgets stated details.
POOR: Treats each turn independently; ignores details stated earlier.

Value: excellent, good, fair, or poor.
Rationale: Cite specific turns where context was or wasn't retained correctly.
"""

# --- Register in Unity Catalog Prompt Registry ---

PROMPT_NAMES = {
    "system_prompt": SYSTEM_PROMPT_TEMPLATE,
    "judge_coherence": COHERENCE_JUDGE_TEMPLATE,
    "judge_context_retention": CONTEXT_RETENTION_JUDGE_TEMPLATE,
}

for name, template in PROMPT_NAMES.items():
    fqn = f"{PROMPT_PREFIX}.{name}"
    mlflow.genai.register_prompt(
        fqn,
        template,
        commit_message=f"Casper's Kitchen: {name}",
    )
    print(f"Registered: {fqn}")

print(f"\nAll prompts registered under {PROMPT_PREFIX}.*")

## Step 2: Define the Casper's Kitchen Support Agent

The agent loads its system prompt from the registry at startup using
`mlflow.genai.load_prompt()`. This means you can edit the prompt in the MLflow UI
and re-run the conversation without changing code.

The key MLflow integration points:
- `@mlflow.trace()` on `handle_message()` creates a trace for each turn
- `mlflow.update_current_trace()` tags each trace with a **session ID** for grouping

In [0]:
import os
from openai import OpenAI
import mlflow
from mlflow.entities import SpanType


class CaspersKitchenAgent:
    """Multi-turn support agent with MLflow session tracking."""

    def __init__(self, model: str):
        self.model = model
        self.client = OpenAI(
            api_key=os.environ["DATABRICKS_TOKEN"],
            base_url=f"{os.environ['DATABRICKS_HOST']}/serving-endpoints",
        )
        self.session_histories: dict[str, list[dict]] = {}

        # Load system prompt from the MLflow Prompt Registry
        self.system_prompt = mlflow.genai.load_prompt(
            f"{PROMPT_PREFIX}.system_prompt", version=1
        ).template
        print(f"Loaded system prompt from registry: {PROMPT_PREFIX}.system_prompt")

    @mlflow.trace(span_type=SpanType.CHAT_MODEL, name="handle_message")
    def handle_message(self, message: str, session_id: str) -> str:
        """Handle one conversation turn with MLflow session tracking."""
        # Tag this trace with the session ID for session-level evaluation
        mlflow.update_current_trace(
            metadata={"mlflow.trace.session": session_id}
        )

        if session_id not in self.session_histories:
            self.session_histories[session_id] = []
        history = self.session_histories[session_id]

        # Build messages: system prompt + full history + new user message
        messages = [{"role": "system", "content": self.system_prompt}]
        messages += history
        messages.append({"role": "user", "content": message})

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            max_completion_tokens=500,
        )
        reply = response.choices[0].message.content.strip()

        # Persist user + assistant exchange in session history
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": reply})
        return reply

    def run_conversation(self, messages: list[str], session_id: str) -> list[str]:
        """Run a full multi-turn conversation, printing each turn."""
        replies = []
        for i, msg in enumerate(messages, 1):
            print(f"\nTurn {i}/{len(messages)}")
            print(f"  Customer: {msg}")
            reply = self.handle_message(msg, session_id)
            print(f"  Agent:    {reply}")
            replies.append(reply)
        return replies

print("Agent class defined.")

## Step 3: Define Conversation Scenarios

Each scenario is a scripted multi-turn conversation that tests specific agent behaviors.

| Scenario | What it tests |
|---|---|
| **Late Catering Order** | Retains order number, address, headcount, and repeat-complaint context across 4 turns |
| **Wrong Items + Allergy** | Retains allergy info (nut allergy) stated in turn 2 and applies it in turns 3-4 without being reminded |

In [0]:
def get_scenario_late_order():
    """Late delivery scenario: tests retention of order details, address, and repeat complaint."""
    return {
        "name": "Late Catering Order",
        "session_id": "session-late-order-001",
        "expected_coherence": True,
        "expected_retention": "excellent",
        "messages": [
            "My catering order for today's office lunch hasn't arrived. Order number CK-2847.",
            "It was supposed to arrive at 11:30 AM and it's now 12:15. We have 30 people waiting.",
            "Can you check if the driver is on the way? We're at 500 Market Street, Suite 400.",
            "Okay, what kind of compensation can you offer? This is the second time this has happened.",
        ],
    }


def get_scenario_wrong_items_allergy():
    """Wrong items + allergy scenario: tests retention of nut allergy stated once in turn 2."""
    return {
        "name": "Wrong Items + Allergy Concern",
        "session_id": "session-wrong-items-001",
        "expected_coherence": True,
        "expected_retention": "excellent",
        "messages": [
            "We just received our catering order CK-3012 but several items are wrong.",
            "We ordered the Mediterranean platter but got the Asian fusion instead. Also, one of our team members has a severe nut allergy.",
            "Can you confirm if the Asian fusion platter contains any nuts? This is urgent.",
            "We need a replacement order with the correct Mediterranean platter. Can you make sure it's nut-free?",
        ],
    }


SCENARIOS = [get_scenario_late_order(), get_scenario_wrong_items_allergy()]

for s in SCENARIOS:
    print(f"{s['name']}: {len(s['messages'])} turns")

## Step 4: Run Conversations

Each scenario runs inside its own `mlflow.start_run()`. Every `handle_message()` call
creates a traced turn tagged with the session ID. We store the `run_id` for evaluation later.

In [0]:
agent = CaspersKitchenAgent(model=AGENT_MODEL)

run_ids = {}

for scenario in SCENARIOS:
    print(f"\n{'=' * 60}")
    print(f"Scenario: {scenario['name']}")
    print(f"Session:  {scenario['session_id']}")
    print(f"{'=' * 60}")

    with mlflow.start_run(run_name=scenario["name"]) as run:
        agent.run_conversation(scenario["messages"], scenario["session_id"])
        run_ids[scenario["session_id"]] = run.info.run_id

    print(f"\nRun ID: {run_ids[scenario['session_id']]}")

print(f"\nAll conversations complete.")

---

# Evaluation: MLflow Session-Level Judges

We evaluate each conversation along **two dimensions** using session-level judges:

| Judge | Output type | What it measures |
|---|---|---|
| `conversation_coherence` | `bool` | Does the conversation flow logically across turns? |
| `context_retention` | `excellent / good / fair / poor` | Does the agent remember details from earlier turns? |

Both judges use the `{{ conversation }}` template variable. This tells MLflow to **aggregate
all traces in the session** and pass the full conversation to each judge, rather than
evaluating individual turns.

## Step 5: Define Session-Level Judges

The judges load their instructions from the Prompt Registry — the same prompts we
registered in Step 1. The `{{ conversation }}` placeholder tells MLflow to:
1. Group all traces by session ID
2. Format them as a conversation transcript
3. Inject the transcript into the judge prompt

In [0]:
from typing import Literal
from mlflow.genai.judges import make_judge

# Load judge instructions from the Prompt Registry
coherence_instructions = mlflow.genai.load_prompt(
    f"{PROMPT_PREFIX}.judge_coherence", version=1
).template
print(f"Loaded: {PROMPT_PREFIX}.judge_coherence")

context_retention_instructions = mlflow.genai.load_prompt(
    f"{PROMPT_PREFIX}.judge_context_retention", version=1
).template
print(f"Loaded: {PROMPT_PREFIX}.judge_context_retention")

# Create judges
judge_model_uri = f"databricks:/{JUDGE_MODEL}"

coherence_judge = make_judge(
    name="conversation_coherence",
    model=judge_model_uri,
    instructions=coherence_instructions,
    feedback_value_type=bool,
)

context_judge = make_judge(
    name="context_retention",
    model=judge_model_uri,
    instructions=context_retention_instructions,
    feedback_value_type=Literal["excellent", "good", "fair", "poor"],
)

# Register judges with the current experiment
coherence_judge.register()
context_judge.register()

print(f"\nJudge model URI: {judge_model_uri}")
print(f"Coherence judge session-level:        {coherence_judge.is_session_level_scorer}")
print(f"Context retention judge session-level: {context_judge.is_session_level_scorer}")
print("Judges registered with experiment.")

## Step 6: Search Traces with `mlflow.search_traces()`

Each conversation turn created a trace tagged with a session ID. We retrieve them
using `mlflow.search_traces()` -- this is the data we'll pass to `evaluate()`.

In [0]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

experiment = mlflow.get_experiment_by_name(f"/Users/{USER_EMAIL}/{EXPERIMENT_NAME}")

scenario_traces = {}

for scenario in SCENARIOS:
    session_id = scenario["session_id"]
    traces = mlflow.search_traces(
        locations=[experiment.experiment_id],
        filter_string=f"run_id = '{run_ids[session_id]}'",
    )
    scenario_traces[session_id] = traces

    print(f"\n{'=' * 50}")
    print(f"{scenario['name']}: {len(traces)} traces")
    print(f"{'=' * 50}")
    display(
        traces[["request_time", "request", "response"]]
        .sort_values(by="request_time", ascending=True)
    )

## Step 7: Evaluate with `mlflow.genai.evaluate()`

This is where the session-level judges receive the **full conversation** via the
`{{ conversation }}` template and score each scenario holistically.

In [0]:
all_results = {}

for scenario in SCENARIOS:
    session_id = scenario["session_id"]
    traces = scenario_traces[session_id]

    print(f"\nEvaluating '{scenario['name']}' ({len(traces)} traces)...")

    with mlflow.start_run(run_name=f"eval-{scenario['name']}") as run:
        eval_results = mlflow.genai.evaluate(
            data=traces,
            scorers=[coherence_judge, context_judge],
        )

    all_results[session_id] = {
        "scenario": scenario["name"],
        "eval_results": eval_results,
        "num_traces": len(traces),
    }
    print(f"  Done. Run ID: {run.info.run_id}")

print("\nAll evaluations complete.")

## Step 8: Inspect Results

In [0]:
for session_id, info in all_results.items():
    result_df = info["eval_results"].result_df

    def extract(keyword, suffix):
        cols = [
            c for c in result_df.columns
            if keyword in c.lower() and suffix in c.lower()
        ]
        if not cols:
            return None
        series = result_df[cols[0]].dropna()
        return series.iloc[0] if len(series) > 0 else None

    def extract_reason(keyword):
        val = extract(keyword, "/reason")
        return val if val is not None else (extract(keyword, "/justification") or "")

    coh_val = extract("coherence", "/value")
    coh_reason = extract_reason("coherence")
    ctx_val = extract("context", "/value")
    ctx_reason = extract_reason("context")

    print(f"\n{'=' * 50}")
    print(f"Scenario: {info['scenario']}")
    print(f"Session:  {session_id}")
    print(f"Traces:   {info['num_traces']}")
    print(f"{'=' * 50}")
    print(f"  Coherence:         {'PASS' if coh_val else 'FAIL'}  ({coh_val})")
    if coh_reason:
        print(f"    {coh_reason}")
    print(f"  Context Retention: {str(ctx_val).upper()}")
    if ctx_reason:
        print(f"    {ctx_reason}")

In [0]:
print("View results in the MLflow Experiment UI:")
print(f"  Experiment: /Users/{USER_EMAIL}/{EXPERIMENT_NAME}")
print(f"  Navigate to: Experiments > {EXPERIMENT_NAME} > Chat Sessions")
for scenario in SCENARIOS:
    print(f"    Session: {scenario['session_id']}")

---

## What Just Happened?

1. **`mlflow.genai.register_prompt()`** stored all prompts (system + 2 judges) in the
   Unity Catalog-backed Prompt Registry with versioning. You can edit them in the MLflow UI
   and re-run without changing code.

2. **`CaspersKitchenAgent`** loaded its system prompt from the registry and ran two 4-turn
   conversations, each traced with `@mlflow.trace()`.

3. **`mlflow.update_current_trace()`** tagged every trace with a session ID, allowing
   MLflow to group all turns as one conversation.

4. **`make_judge()`** created two session-level judges whose instructions were also loaded
   from the registry. The `{{ conversation }}` template variable is what makes them
   session-level -- MLflow aggregates all turns before scoring.

5. **`mlflow.genai.evaluate()`** sent the full conversation to each judge for holistic scoring.

### Key Takeaway: Context Retention

The **Wrong Items + Allergy** scenario is the harder test. The customer mentions a nut allergy
once in turn 2. In turns 3 and 4, the agent must carry that constraint forward without being
reminded -- the context retention judge specifically evaluates whether this happens.